
# Projekt: Dyfuzja ciepła w pręcie — numeryczne modelowanie jednorodne i niejednorodne

**Temat:** Numeryczna symulacja przepływu ciepła w pręcie oraz analiza wpływu zmiennych parametrów materiałowych (niejednorodności) na rozkład temperatury.

Pomysł jest prosty i ma bezpośrednie przełożenie inżynierskie: badamy jednowymiarowy pręt, którego końce utrzymywane są w stałych, skrajnych temperaturach (np. 100°C z jednej strony i 0°C z drugiej), a wewnątrz nakładamy narzucony profil początkowy. Pręt może być lity (jednorodny, np. wykonany w całości ze stali) lub złożony z różnych metali (niejednorodny, np. zespawane ze sobą odcinki stali i miedzi). Chcemy policzyć, jak temperatura rozkłada się i ewoluuje w czasie wewnątrz pręta, w sytuacjach, gdzie klasyczna matematyka analityczna przestaje wystarczać.

Notebook pokazuje:

* **Model fizyczny i założenia** dla przewodnictwa cieplnego (równania w postaci różniczkowej i konserwatywnej).
* **Jawną metodę Eulera** przetestowaną na materiale jednorodnym.
* **Warunek stabilności Couranta-Friedrichsa-Lewy'ego (CFL)** i jaskrawy przykład tego, co idzie źle (eksplozja numeryczna), gdy go złamiemy.
* **Bezwarunkowo stabilną metodę Cranka–Nicolsona** opartą na rozwiązywaniu układów równań liniowych z macierzami rzadkimi.
* **Symulację pręta niejednorodnego** z wykorzystaniem średniej harmonicznej na styku różnych materiałów (stal vs miedź).
* **Podejście spektralne (FFT)** w dziedzinie częstotliwości jako wysoce precyzyjny punkt odniesienia.
* **Zestawienie algorytmów numerycznych** i ich złożoności obliczeniowej (od $\mathcal{O}(N)$ do $\mathcal{O}(N \log N)$).
* **Czystą, wektorową implementację** algorytmów w języku Julia z wykorzystaniem pakietów `LinearAlgebra`, `SparseArrays` oraz `FFTW`.

---

**Cel wersji uzupełnionej:** notebook został uporządkowany tak, aby odpowiadał wymaganiom projektu z obliczeń naukowych: pokazuje model, założenia, algorytmy, złożoność, alternatywy, kluczowe fragmenty kodu, analizę dokładności i wydajności, wizualizację oraz znaczenie praktyczne wyników. Część kodowa może działać samodzielnie w Jupyterze, a część pakietowa powinna być utrzymywana w repozytorium Julia na GitHubie.


## 0. Zgodność z wymaganiami projektu

Wymagania projektu zostały rozpisane bezpośrednio na elementy notebooka i repozytorium.

| Wymaganie | Gdzie jest realizowane w notebooku/repo |
|---|---|
| Omówienie założeń | sekcje 1 i 1A |
| Demonstracja, co może pójść źle | sekcje o CFL, tłumieniu numerycznym i oscylacjach Cranka--Nicolsona |
| Opis algorytmów | sekcje 5, 6 i 6A |
| Złożoność i alternatywy | sekcja 6A |
| Opis kluczowych fragmentów kodu | sekcja 6B oraz komentarze w funkcjach |
| Analiza dokładności i wydajności | sekcja 7 |
| Wizualizacja wyników | wykresy profilu materiałowego, przebiegu temperatury, heatmapa i animacje |
| Praktyczne znaczenie | sekcja 8 |
| Pakiet Julia na GitHubie | foldery `src/`, `test/`, `Project.toml`, `README.md` oraz workflow CI |
| Dokumentacja eksportowanych metod | sekcja 9 zawiera listę funkcji, które powinny mieć docstringi w `src/RodHeatDiffusion.jl` |

Dzięki temu notebook może pełnić funkcję prezentacji/sprawozdania, a repozytorium pozostaje miejscem na czysty kod pakietu i testy.

## 1. Model matematyczny

Dla jednowymiarowego pręta rozważamy ogólne równanie przewodnictwa ciepła, które pozwala uwzględnić właściwości materiału zależne od położenia:

$$
\rho(x)c(x)\frac{\partial T}{\partial t}
=
\frac{\partial}{\partial x}\left(k(x)\frac{\partial T}{\partial x}\right),
$$

gdzie:

- $T(x,t)$ — temperatura w pręcie,
- $k(x)$ — przewodność cieplna materiału $[W/(mK)]$,
- $\rho(x)$ — gęstość $[kg/m^3]$,
- $c(x)$ — ciepło właściwe $[J/(kgK)]$.

W przypadku **pręta jednorodnego** (np. wykonanego w całości z jednego metalu), parametry te są stałe, a równanie redukuje się do klasycznej postaci:

$$
\frac{\partial T}{\partial t}=\alpha \frac{\partial^2 T}{\partial x^2},
\qquad
\alpha=\frac{k}{\rho c}.
$$

W projekcie przyjmujemy warunki brzegowe Dirichleta, wymuszając stałą temperaturę na końcach pręta:

$$
T(0,t)=T_{lewy}, \qquad T(L,t)=T_{prawy}.
$$

Interpretacja:

- lewy koniec pręta utrzymywany jest w zadanej temperaturze (np. ogrzewany do $100^\circ C$),
- prawy koniec pręta utrzymywany jest w zadanej temperaturze (np. chłodzony do $0^\circ C$).

## 1A. Założenia modelu i co może pójść źle

Model jest celowo prosty, ale każde uproszczenie ma konsekwencje numeryczne.

### Przyjęte założenia

1. **Pręt jest jednowymiarowy.** Temperatura zależy tylko od położenia $x$ i czasu $t$. Pomijamy zmiany w przekroju poprzecznym.
2. **Parametry materiałowe są stałe w obrębie segmentu.** Dopuszczamy skok parametrów na granicy stal--miedź, ale nie opisujemy zmian zależnych od temperatury.
3. **Brzegi mają narzuconą temperaturę Dirichleta.** Końce pręta są traktowane jak idealne termostaty.
4. **Brak źródeł ciepła wewnątrz pręta.** Analizujemy samo przewodzenie i wygaszanie profilu temperatury.
5. **Siatka jest jednorodna.** Odległość między punktami wynosi stałe $\Delta x$.

### Przykłady naruszeń założeń

- Jeśli dla metody jawnej wybierzemy zbyt duży krok $\Delta t$, rozwiązanie może eksplodować lub zacząć oscylować. To jest klasyczny błąd naruszenia warunku CFL.
- Jeśli warunek początkowy jest niespójny z warunkiem brzegowym, np. cały pręt ma 1000°C, ale końce nagle ustawiamy na 0°C, w metodzie Cranka--Nicolsona mogą pojawić się niefizyczne oscylacje.
- Jeśli na granicy materiałów użyjemy zwykłej średniej arytmetycznej zamiast harmonicznej, strumień ciepła na styku materiałów może być źle aproksymowany.
- Jeśli użyjemy bardzo dużego kroku w Eulerze niejawnym, metoda pozostanie stabilna, ale wynik będzie nadmiernie wygładzony i zbyt mocno tłumiony.

Te przypadki są pokazane dalej jako eksperymenty numeryczne, bo w projekcie nie wystarczy podać algorytm - trzeba też pokazać, kiedy i dlaczego może dawać złe wyniki.


## 2. Instalacja i import pakietów

Ten notebook używa głównie standardowych bibliotek Julii. Do wykresów używany jest `Plots.jl`.


In [ ]:

using LinearAlgebra
using SparseArrays
using Printf

try
    using Plots
    using FFTW
catch
    import Pkg
    Pkg.add("Plots")
    Pkg.add("FFTW")
    using Plots
    using FFTW
end

default(size=(900, 520), linewidth=2, margin=5Plots.mm)


## 3. Dane materiałowe

Wartości poniżej są orientacyjne i służą do projektu numerycznego. Dla pracy końcowej można je zastąpić wartościami z norm, podręczników lub kart materiałowych producentów.


In [ ]:
Base.@kwdef struct Segment
    name::String
    length::Float64      # m (zamiast grubości ściany)
    k::Float64           # W/(m K) - przewodność cieplna
    rho::Float64         # kg/m^3 - gęstość
    c::Float64           # J/(kg K) - ciepło właściwe
end

function print_segments(segments)
    println("Struktura pręta:")
    for s in segments
        @printf("%-15s długość = %.3f m, k = %6.2f W/(mK), rho*c = %.2e J/(m³K)\n",
                s.name, s.length, s.k, s.rho*s.c)
    end
    
    # Opór cieplny (zastępczy, zakłada pole przekroju 1 m^2)
    R = sum(s.length / s.k for s in segments)
    @printf("\nCałkowity opór cieplny R = %.5f m²K/W\n", R)
end

rod_homogeneous = [
    Segment(name="Stal", length=1.0, k=50.0, rho=7800.0, c=450.0)
]

rod_heterogeneous = [
    Segment(name="Stal",  length=0.5, k=50.0,  rho=7800.0, c=450.0),
    Segment(name="Miedź", length=0.5, k=400.0, rho=8900.0, c=390.0)
]

print_segments(rod_heterogeneous)



## 4. Budowa siatki obliczeniowej

Dzielimy pręt na $N_x$ punktów. Każdemu punktowi przypisujemy materiał, czyli wartości $k$ oraz $\rho c$.

In [ ]:
function build_rod(segments::Vector{Segment}; Nx::Int=151)
    L = sum(s.length for s in segments)
    x = collect(range(0.0, L; length=Nx))
    edges = cumsum([s.length for s in segments])

    k = zeros(Float64, Nx)
    rhoc = zeros(Float64, Nx)
    names = Vector{String}(undef, Nx)

    for (i, xi) in enumerate(x)
        # +eps pomaga poprawnie przypisać punkt x=0 do pierwszego segmentu
        j = searchsortedfirst(edges, xi + eps(Float64))
        j = clamp(j, 1, length(segments))
        segment = segments[j]
        k[i] = segment.k
        rhoc[i] = segment.rho * segment.c
        names[i] = segment.name
    end

    return x, k, rhoc, names
end

x, k, rhoc, names = build_rod(rod_heterogeneous; Nx=151)

plot(x, k,
     xlabel="x [m]",
     ylabel="k [W/(mK)]",
     title="Przewodność cieplna wzdłuż pręta (Stal + Miedź)",
     label="k(x)",
     color=:orange)

## 5. Dyskretyzacja operatora przewodzenia

Dla pręta o właściwościach zmiennych w przestrzeni (niejednorodnego) najwygodniejsza jest postać konserwatywna:

$$\rho_i c_i \frac{dT_i}{dt}=\frac{1}{\Delta x^2}\left[k_{i+1/2}(T_{i+1}-T_i)-k_{i-1/2}(T_i-T_{i-1})\right]$$

Na granicach między różnymi materiałami pręta (np. na spawie stali z miedzią) używamy średniej harmonicznej:

$$k_{i+1/2}=\frac{2k_i k_{i+1}}{k_i+k_{i+1}}$$

Jest to znacznie sensowniejsze w inżynierii niż zwykła średnia arytmetyczna, ponieważ na styku dwóch metali głównym dławikiem dla przepływu ciepła staje się segment o mniejszej przewodności.

In [ ]:
harmonic_mean(a, b) = 2a*b/(a+b)

# Obliczanie przewodności na styku komórek (np. na spawie stali z miedzią)
function face_conductivity(k)
    return [harmonic_mean(k[i], k[i+1]) for i in 1:length(k)-1]
end

# Budowa rzadkiej macierzy trójprzekątniowej dla pręta niejednorodnego
function build_operator(k, rhoc, dx)
    N = length(k)
    n = N - 2                       # liczba punktów wewnętrznych pręta (bez brzegów)
    kh = face_conductivity(k)

    rows = Int[]
    cols = Int[]
    vals = Float64[]

    leftcoef = zeros(Float64, n)    # wpływ lewego końca pręta (np. 100°C)
    rightcoef = zeros(Float64, n)   # wpływ prawego końca pręta (np. 0°C)

    for j in 1:n
        i = j + 1                   # indeks globalny wzdłuż pręta
        aL = kh[i-1] / (rhoc[i] * dx^2)
        aR = kh[i]   / (rhoc[i] * dx^2)

        # Główna przekątna
        push!(rows, j); push!(cols, j); push!(vals, -(aL + aR))

        # Lewy sąsiad lub lewy brzeg
        if j > 1
            push!(rows, j); push!(cols, j-1); push!(vals, aL)
        else
            leftcoef[j] = aL
        end

        # Prawy sąsiad lub prawy brzeg
        if j < n
            push!(rows, j); push!(cols, j+1); push!(vals, aR)
        else
            rightcoef[j] = aR
        end
    end

    # Utworzenie macierzy rzadkiej w celu optymalizacji pamięci i czasu
    L_matrix = sparse(rows, cols, vals, n, n)
    return L_matrix, leftcoef, rightcoef
end


# --- Weryfikacja działania funkcji na naszej siatce pręta ---
dx = x[2] - x[1]
Lop, leftcoef, rightcoef = build_operator(k, rhoc, dx)

println("Zbudowano macierz rzadką (Sparse Matrix) o rozmiarze: ", size(Lop))



## 6. Metody numeryczne

### 6.1 Jawna metoda Eulera

Schemat:

$$
T^{n+1}=T^n+\Delta t\,L T^n.
$$

Zaleta: bardzo prosty i tani krok czasowy.

Wada: metoda jest stabilna tylko dla odpowiednio małego kroku czasu.

### 6.2 Niejawna metoda Eulera

Schemat:

$$
(I-\Delta t L)T^{n+1}=T^n.
$$

Zaleta: stabilna dla dużych kroków czasu.

Wada: w każdym kroku rozwiązujemy układ równań liniowych.

### 6.3 Crank–Nicolson

Schemat:

$$
(I-\frac{\Delta t}{2}L)T^{n+1}=(I+\frac{\Delta t}{2}L)T^n.
$$

Zaleta: dokładniejszy w czasie niż obie metody Eulera.


## 6A. Złożoność obliczeniowa i alternatywy

Dla siatki o $N$ punktach wewnętrznych operator przewodzenia ma postać macierzy trójprzekątniowej/rzadkiej. To jest ważne, bo nie przechowujemy pełnej macierzy $N \times N$.

| Metoda | Koszt jednego kroku | Pamięć | Stabilność | Komentarz |
|---|---:|---:|---|---|
| Euler jawny | $O(N)$ | $O(N)$ | warunkowa | tani krok, ale często bardzo dużo kroków |
| Euler niejawny | $O(N)$ po faktoryzacji | $O(N)$ | bezwarunkowa dla problemu liniowego | trzeba rozwiązywać układ równań |
| Crank--Nicolson | $O(N)$ po faktoryzacji | $O(N)$ | zwykle stabilny, ale może oscylować dla ostrych danych | dobry kompromis dokładności i kosztu |

W implementacji używamy `SparseArrays` i `factorize(A)`. Dzięki temu macierz nie jest traktowana jak gęsta, co dla dużych siatek miałoby koszt pamięci $O(N^2)$.

### Możliwe alternatywy

- **Metoda elementów skończonych (FEM)** - lepsza dla geometrii 2D/3D lub nieregularnych siatek.
- **Metody objętości skończonych (FVM)** - bardzo naturalne dla praw zachowania i nieciągłości materiałowych.
- **Metody spektralne** - bardzo dokładne dla gładkich rozwiązań, ale mniej wygodne przy skokowych parametrach materiałowych.
- **Adaptacyjny krok czasu** - automatycznie zmniejsza $\Delta t$ przy ostrych gradientach, a zwiększa w spokojnych fazach symulacji.

In [ ]:
using LinearAlgebra
using SparseArrays

# 1. Średnia harmoniczna
harmonic_mean(a, b) = 2a*b/(a+b)

function face_conductivity(k)
    return [harmonic_mean(k[i], k[i+1]) for i in 1:length(k)-1]
end

# 2. Budowa macierzy dla pręta
function build_operator(k, rhoc, dx)
    N = length(k)
    n = N - 2
    kh = face_conductivity(k)

    rows, cols, vals = Int[], Int[], Float64[]
    leftcoef, rightcoef = zeros(n), zeros(n)

    for j in 1:n
        i = j + 1
        aL = kh[i-1] / (rhoc[i] * dx^2)
        aR = kh[i]   / (rhoc[i] * dx^2)

        push!(rows, j); push!(cols, j); push!(vals, -(aL + aR))
        if j > 1 push!(rows, j); push!(cols, j-1); push!(vals, aL) else leftcoef[j] = aL end
        if j < n push!(rows, j); push!(cols, j+1); push!(vals, aR) else rightcoef[j] = aR end
    end
    return sparse(rows, cols, vals, n, n), leftcoef, rightcoef
end

# 3. Limit stabilności CFL
function explicit_dt_limit(k, rhoc, dx; safety=0.8)
    kh = face_conductivity(k)
    rates = [(kh[i-1] + kh[i]) / (rhoc[i] * dx^2) for i in 2:length(k)-1]
    return safety / maximum(rates)
end

# 4. Główny uniwersalny solver (jawny i niejawny)
function solve_rod_theta(x, k, rhoc; T_initial, T_left, T_right, dt, t_end, theta, save_every=1)
    dx = x[2] - x[1]
    Lop, leftcoef, rightcoef = build_operator(k, rhoc, dx)
    Id = spdiagm(0 => ones(length(x)-2))

    A = Id - theta * dt * Lop
    B = Id + (1 - theta) * dt * Lop
    F = factorize(A)

    T = copy(T_initial)
    
    # Rozpoznawanie, czy warunek brzegowy jest funkcją czasu t, czy stałą liczbą
    get_T_left(t)  = T_left isa Function ? T_left(t) : T_left
    get_T_right(t) = T_right isa Function ? T_right(t) : T_right
    
    T[1] = get_T_left(0.0)
    T[end] = get_T_right(0.0)
    
    nt = ceil(Int, t_end/dt)
    times = Float64[]
    saved = Vector{Vector{Float64}}()

    for n in 0:nt
        t = n * dt
        # Zapisywanie rozwiązania tylko co określoną liczbę kroków
        if n % save_every == 0
            push!(times, t)
            push!(saved, copy(T))
        end
        n == nt && break
        
        b_now = leftcoef .* get_T_left(t) .+ rightcoef .* get_T_right(t)
        b_next = leftcoef .* get_T_left(t + dt) .+ rightcoef .* get_T_right(t + dt)
        
        rhs = B * T[2:end-1] .+ dt .* ((1 - theta) .* b_now .+ theta .* b_next)
        T[2:end-1] .= F \ rhs
        T[1] = get_T_left(t + dt)
        T[end] = get_T_right(t + dt)
    end
    
    return times, permutedims(hcat(saved...))
end

# 5. Definicje poszczególnych metod
solve_rod_explicit(args...; kwargs...) = solve_rod_theta(args...; theta=0.0, kwargs...)
solve_rod_crank_nicolson(args...; kwargs...) = solve_rod_theta(args...; theta=0.5, kwargs...)
solve_rod_implicit(args...; kwargs...) = solve_rod_theta(args...; theta=1.0, kwargs...)

## 6B. Kluczowe fragmenty kodu

Najważniejsze funkcje projektu są następujące:

| Funkcja | Rola w projekcie |
|---|---|
| `Segment` | przechowuje parametry jednego materiału: długość, przewodność, gęstość i ciepło właściwe |
| `build_rod` | buduje siatkę przestrzenną i przypisuje każdemu punktowi odpowiedni materiał |
| `harmonic_mean` | liczy przewodność efektywną na granicy dwóch komórek |
| `face_conductivity` | tworzy tablicę przewodności na interfejsach komórek |
| `build_operator` | buduje rzadką macierz operatora dyfuzji ciepła |
| `explicit_dt_limit` | wyznacza bezpieczny krok czasu dla metody jawnej |
| `solve_rod_theta` | uniwersalny solver schematu $\theta$ |
| `solve_rod_explicit` | wrapper dla Eulera jawnego, czyli $\theta=0$ |
| `solve_rod_implicit` | wrapper dla Eulera niejawnego, czyli $\theta=1$ |
| `solve_rod_crank_nicolson` | wrapper dla metody Cranka--Nicolsona, czyli $\theta=1/2$ |

W repozytorium te funkcje powinny znajdować się w `src/RodHeatDiffusion.jl`, być eksportowane przez moduł oraz posiadać docstringi. Notebook może je powtarzać dla wygody prezentacji, ale wersją źródłową powinna być implementacja pakietowa.

### SYMULACJA — EULER JAWNY

In [ ]:
x, k_het, rhoc_het, names_het = build_rod(rod_heterogeneous; Nx=151)

T0_het = 200 .* exp.(-((x .- 0.5).^2) ./ (2 * 0.06^2))

dx = x[2] - x[1]

dt_explicit = explicit_dt_limit(k_het, rhoc_het, dx; safety=0.45)

println("Bezpieczny krok czasowy dla jawnego Eulera:")
println("dt = ", dt_explicit, " s")

In [ ]:
using Plots

t_end = 2000.0

# Nie zapisujemy każdej iteracji, bo dt jest bardzo małe.
# Zapisujemy stan mniej więcej co 20 sekund symulacji.
save_every = max(1, ceil(Int, 20.0 / dt_explicit))

times_fe, sol_fe = solve_rod_explicit(
    x, k_het, rhoc_het;
    T_initial = T0_het,
    T_left  = t -> 0.0,
    T_right = t -> 0.0,
    dt = dt_explicit,
    t_end = t_end,
    save_every = save_every
)

anim = @animate for idx in 1:length(times_fe)
    plot(
        x,
        sol_fe[idx, :],
        linewidth = 3,
        label = "T(x,t)",
        xlabel = "położenie x [m]",
        ylabel = "temperatura [°C]",
        title = "Euler jawny, pręt stal–miedź, t = $(round(Int, times_fe[idx])) s",
        ylim = (0, 205),
        legend = :topright
    )

    vline!(
        [0.5],
        linestyle = :dash,
        linewidth = 2,
        label = "granica stal/miedź"
    )

    annotate!(0.25, 90, text("stal", 10))
    annotate!(0.75, 90, text("miedź", 10))
end

gif(anim, "pret_stal_miedz_euler_jawny.gif", fps = 5)

## SYMULACJE

In [ ]:
using Plots

# Symulacja: temperatura początkowo skupiona w środku pręta
x, k_het, rhoc_het, names_het = build_rod(rod_heterogeneous; Nx=151)

T0_het = 100 .* exp.(-((x .- 0.5).^2) ./ (2 * 0.06^2))

dt = 100.0
t_end = 2000.0

times_be, sol_be = solve_rod_implicit(
    x, k_het, rhoc_het;
    T_initial = T0_het,
    T_left  = t -> 0.0,
    T_right = t -> 0.0,
    dt = dt,
    t_end = t_end,
    save_every = 1
)

# Animacja GIF
anim = @animate for idx in 1:length(times_be)
    plot(
        x,
        sol_be[idx, :],
        linewidth = 3,
        label = "T(x,t)",
        xlabel = "położenie x [m]",
        ylabel = "temperatura [°C]",
        title = "Pręt stal–miedź, t = $(round(Int, times_be[idx])) s",
        ylim = (0, 105),
        legend = :topright
    )

    vline!(
        [0.5],
        linestyle = :dash,
        linewidth = 2,
        label = "granica stal/miedź"
    )

    annotate!(0.25, 90, text("stal", 10))
    annotate!(0.75, 90, text("miedź", 10))
end

gif(anim, "pret_stal_miedz_temperatura.gif", fps = 5)

In [ ]:
using Plots

# Ten sam warunek początkowy dla wszystkich testów
T0_test = 100 .* exp.(-((x .- 0.5).^2) ./ (2 * 0.05^2))

dt_values = [1.0, 20.0, 200.0]
t_end = 1000.0

p = plot(
    xlabel = "czas [s]",
    ylabel = "maksymalna temperatura [°C]",
    title = "Euler niejawny: zbyt duży krok czasu tłumi rozwiązanie",
    legend = :topright
)

for dt in dt_values
    times, sol = solve_rod_implicit(
        x, k_het, rhoc_het;
        T_initial = T0_test,
        T_left  = t -> 0.0,
        T_right = t -> 0.0,
        dt = dt,
        t_end = t_end,
        save_every = 1
    )

    Tmax = [maximum(sol[i, :]) for i in 1:size(sol, 1)]

    plot!(
        p,
        times,
        Tmax,
        linewidth = 3,
        marker = :circle,
        markersize = 3,
        label = "Δt = $(dt) s"
    )
end

display(p)

### SYMULACJE CRANK NICOLSON

In [ ]:
N = 100 
L = sum(s.length for s in rod_heterogeneous) #calkowita długość pręta
x = collect(range(0, L, length=N))

#Pułapka: niejednorodność
k_rod = zeros(N)
rhoc_rod = zeros(N)

for i in 1:N
    #lewa - stal, prawa miedz
    if x[i] <= rod_heterogeneous[1].length
        k_rod[i] = rod_heterogeneous[1].k
        rhoc_rod[i] = rod_heterogeneous[1].rho * rod_heterogeneous[1].c
    else
        k_rod[i] = rod_heterogeneous[2].k
        rhoc_rod[i] = rod_heterogeneous[2].rho * rod_heterogeneous[2].c
    end
end

In [ ]:
#PUŁAPKA: Niespójność warunków brzegowych
#rozgrzewamy pręt do 1000°C i natychmiastowo stykamy brzegi z termostatem 0°C.

T_initial = fill(1000.0, N)
T_initial[1] = 0.0   # Gwałtowny skok (brzeg lewy)
T_initial[end] = 0.0 # Gwałtowny skok (brzeg prawy)

t_end_exp = 200.0 # Całkowity czas symulacji (s)

# Funkcja pomocnicza potrzebna do nowego solvera z grupy (jeśli nie jest zdefiniowana wyżej)
if !@isdefined(boundary_vector)
    boundary_vector(leftcoef, rightcoef, TL, TR) = leftcoef .* TL .+ rightcoef .* TR
end

#EKSPERYMENT 1: Zbyt duży krok czasowy -> Oscylacje numeryczne
dt_bad = 10.0 #sek
times_bad, results_bad = solve_rod_crank_nicolson(x, k_rod, rhoc_rod, 
                                                   T_initial=T_initial, 
                                                   T_left = t -> 0.0, T_right = t -> 0.0, 
                                                   dt=dt_bad, t_end=t_end_exp)

#EKSPERYMENT 2: Poprawnie dobrany krok czasowy
dt_good = 0.5 #sek
times_good, results_good = solve_rod_crank_nicolson(x, k_rod, rhoc_rod, 
                                                     T_initial=T_initial, 
                                                     T_left = t -> 0.0, T_right = t -> 0.0, 
                                                     dt=dt_good, t_end=t_end_exp, save_every=4)


idx_bad = argmin(abs.(times_bad .- 10.0))
idx_good = argmin(abs.(times_good .- 10.0))

p1 = plot(x, results_bad[idx_bad, :], label="Crank-Nicolson (zly dt = 10s)", 
          title="Oscylacje numeryczne pułapka niespójności",
          xlabel="położenie na pręcie [m]", ylabel="temperatura [°C]",
          color=:red, linestyle=:dash, marker=:circle, markersize=3)

plot!(p1, x, results_good[idx_good, :], label="Crank-Nicolson (dobry dt = 0.5s)", color=:blue, linewidth=3)
vline!(p1, [0.5], label = "", color=:black, linestyle=:dot)
display(p1)

In [ ]:
#Heatmapa asymetrycznego przepływu ciepła
p2 = heatmap(x, times_good, results_good, 
             title="Dyfuzja ciepła: Stal + Miedź", 
             xlabel="położenie [m]", ylabel="czas [s]", 
             color=:inferno, right_margin=20Plots.mm)

vline!(p2, [0.5], label="", color=:black, linestyle=:dash, linewidth=2)
annotate!(p2, 0.25, t_end_exp * 0.9, text("STAL\n(Wolne stygnięcie)", :black, 10, :center))
annotate!(p2, 0.75, t_end_exp * 0.9, text("MIEDŹ\n(Szybkie stygnięcie)", :black, 10, :center))
display(p2)

In [ ]:
#Animacja stygnięcia pręta niejednorodnego
frames_to_render = 1:1:size(results_good, 1)

anim = @animate for i in frames_to_render
    current_time = times_good[i]

    p_anim = plot(x, results_good[i, :], 
                  title = @sprintf("Metoda Crank-Nicolson\nczas: %.1f s", current_time),
                  xlabel = "położenie na pręcie [m]", 
                  ylabel = "temperatura [°C]",
                  label = "rozkład temperatury T(x)",
                  color = :firebrick, 
                  linewidth = 4,
                  ylim = (-50, 1050),
                  legend = :bottomleft)

    vline!(p_anim, [0.5], label="granica stal/miedź", color=:black, linestyle=:dash, linewidth=2)
    annotate!(p_anim, 0.25, 950, text("STAL", :black, 12))
    annotate!(p_anim, 0.75, 950, text("MIEDŹ", :black, 12))
end

gif_filename = "crank_nicolson_cooling.gif"
gif(anim, gif_filename, fps = 15)
println("Animacja zapisana: ", gif_filename)

## 7. Analiza dokładności i wydajności trzech metod

W projekcie porównujemy trzy warianty schematu $\theta$ dla równania dyfuzji ciepła w pręcie stal–miedź:

- **Euler jawny** — $\theta=0$,
- **Euler niejawny** — $\theta=1$,
- **Crank–Nicolson** — $\theta=\frac12$.

Wszystkie trzy metody korzystają z tej samej dyskretyzacji przestrzennej: rzadkiej macierzy trójprzekątniowej operatora przewodzenia ciepła oraz średniej harmonicznej przewodności na granicach komórek. Dzięki temu różnice między metodami wynikają głównie ze sposobu całkowania po czasie.

### 7.1 Dokładność teoretyczna

| Metoda | Błąd czasowy | Błąd przestrzenny | Stabilność | Główne źródło błędu |
|---|---:|---:|---|---|
| Euler jawny | $O(\Delta t)$ | $O(\Delta x^2)$ | warunkowa, wymaga CFL | niestabilność lub duża liczba bardzo małych kroków |
| Euler niejawny | $O(\Delta t)$ | $O(\Delta x^2)$ | bezwarunkowa dla liniowego równania ciepła | tłumienie numeryczne, czyli sztuczne wygładzanie profilu |
| Crank–Nicolson | $O(\Delta t^2)$ | $O(\Delta x^2)$ | stabilna, ale może oscylować dla ostrych skoków | oscylacje przy zbyt dużym $\Delta t$ i niespójnych danych |

W praktyce oznacza to, że dla gładkiego profilu początkowego Crank–Nicolson powinien dawać najmniejszy błąd przy tym samym kroku czasu. Euler jawny jest poprawny tylko wtedy, gdy spełniony jest warunek CFL

$$
\Delta t \leq \min_i \frac{\rho_i c_i (\Delta x)^2}{k_{i-1/2}+k_{i+1/2}}.
$$

Po przekroczeniu tego limitu pojawiają się oscylacje i wartości niefizyczne. Euler niejawny nie wybucha dla dużego kroku, ale przy zbyt dużym $\Delta t$ nadmiernie wygładza rozwiązanie. Crank–Nicolson jest dokładniejszy, lecz przy nagłych skokach temperatury i zbyt dużym kroku może dawać oscylacje typu Gibbsa.

### 7.2 Wydajność obliczeniowa

Niech $N$ oznacza liczbę punktów siatki. Macierz operatora jest trójprzekątniowa i rzadka, więc pojedyncze mnożenie lub rozwiązanie układu ma koszt liniowy względem liczby punktów.

| Metoda | Koszt przygotowania | Koszt jednego kroku | Liczba kroków | Wniosek wydajnościowy |
|---|---:|---:|---:|---|
| Euler jawny | $O(N)$ | $O(N)$ | bardzo duża, bo $\Delta t=O(\Delta x^2)$ | tani krok, ale słaby dla długich symulacji i gęstych siatek |
| Euler niejawny | $O(N)$ po faktoryzacji macierzy | $O(N)$ | mała, można brać duże $\Delta t$ | dobry do długich symulacji, gdy akceptujemy tłumienie |
| Crank–Nicolson | $O(N)$ po faktoryzacji macierzy | $O(N)$ | średnia | najlepszy kompromis między kosztem i dokładnością |

Dla metody jawnej zagęszczenie siatki jest szczególnie kosztowne: gdy zmniejszymy $\Delta x$ dwukrotnie, limit stabilności wymusza około czterokrotnie mniejszy krok czasu. W metodach niejawnych macierz $A$ jest stała, więc faktoryzujemy ją tylko raz i potem używamy tej samej faktoryzacji w kolejnych krokach.


In [ ]:
using Printf
using Statistics

# Eksperyment porównawczy: dokładność względem bardzo dokładnego rozwiązania referencyjnego
# oraz orientacyjny czas obliczeń. Komórka zakłada, że funkcje solve_rod_explicit,
# solve_rod_implicit i solve_rod_crank_nicolson są zdefiniowane w sekcji 6.

function final_state(solver, x, k, rhoc, T0; dt, t_end)
    nsteps = ceil(Int, t_end / dt)
    times, sol = solver(
        x, k, rhoc;
        T_initial = T0,
        T_left = t -> 0.0,
        T_right = t -> 0.0,
        dt = dt,
        t_end = t_end,
        save_every = max(1, nsteps)
    )
    return sol[end, :], times[end], nsteps
end

function error_norms(T, Tref, dx)
    err = T .- Tref
    L2 = sqrt(sum(abs2, err) * dx)
    Linf = maximum(abs.(err))
    return L2, Linf
end

function benchmark_three_methods(; Nx=101, t_end=50.0)
    x_b, k_b, rhoc_b, _ = build_rod(rod_heterogeneous; Nx=Nx)
    dx_b = x_b[2] - x_b[1]
    T0_b = 100 .* exp.(-((x_b .- 0.5).^2) ./ (2 * 0.06^2))

    # Rozwiązanie referencyjne: Crank--Nicolson z bardzo małym krokiem czasu.
    # Nie traktujemy go jako rozwiązania analitycznego, ale jako praktyczny punkt odniesienia.
    dt_ref = 0.01
    Tref, _, _ = final_state(solve_rod_crank_nicolson, x_b, k_b, rhoc_b, T0_b;
                             dt=dt_ref, t_end=t_end)

    dt_cfl = explicit_dt_limit(k_b, rhoc_b, dx_b; safety=0.45)

    configs = [
        ("Euler jawny", solve_rod_explicit, min(dt_cfl, 0.1)),
        ("Euler niejawny", solve_rod_implicit, 1.0),
        ("Crank-Nicolson", solve_rod_crank_nicolson, 1.0)
    ]

    println("Porównanie dla Nx = $(Nx), t_end = $(t_end) s")
    @printf("Limit CFL dla Eulera jawnego: %.5f s\n\n", dt_cfl)
    @printf("%-18s %10s %10s %14s %14s %12s\n", "Metoda", "dt [s]", "kroki", "błąd L2", "błąd max", "czas [s]")
    println("-"^82)

    results = []
    for (name, solver, dt) in configs
        GC.gc()
        t0 = time()
        Tnum, tfinal, nsteps = final_state(solver, x_b, k_b, rhoc_b, T0_b;
                                           dt=dt, t_end=t_end)
        elapsed = time() - t0
        L2, Linf = error_norms(Tnum, Tref, dx_b)
        push!(results, (name=name, dt=dt, steps=nsteps, L2=L2, Linf=Linf, elapsed=elapsed))
        @printf("%-18s %10.4f %10d %14.6e %14.6e %12.4f\n", name, dt, nsteps, L2, Linf, elapsed)
    end

    return results
end

benchmark_results = benchmark_three_methods()


### 7.3 Interpretacja eksperymentu porównawczego

Wyniki z komórki powyżej należy interpretować następująco:

- **Euler jawny** zwykle wykonuje najwięcej kroków, ponieważ jego $\Delta t$ jest ograniczony warunkiem CFL. Jeżeli krok jest dobrany poprawnie, wynik jest fizyczny, ale koszt rośnie szybko przy zagęszczaniu siatki.
- **Euler niejawny** może wykonać symulację znacznie większym krokiem czasu. Czas obliczeń bywa mniejszy, ale błąd może być większy, bo metoda wprowadza silne tłumienie numeryczne.
- **Crank–Nicolson** ma koszt jednego kroku podobny do Eulera niejawnego, ale wyższy rząd dokładności w czasie. Dla gładkich danych daje najlepszy kompromis: relatywnie mało kroków i mały błąd.

Wniosek praktyczny: metoda jawna jest dobra jako prosta metoda kontrolna i demonstracyjna, Euler niejawny jako metoda odporna dla długich symulacji, a Crank–Nicolson jako główny wybór, gdy zależy nam na dokładności przebiegu przejściowego.


## 8. Omówienie praktycznego znaczenia trzech metod

### 8.1 Euler jawny

Euler jawny ma znaczenie praktyczne przede wszystkim jako metoda szybka, prosta i bardzo przejrzysta. Jeden krok korzysta tylko z aktualnych temperatur w sąsiednich punktach, dlatego łatwo kontrolować przepływ informacji w siatce i łatwo wykrywać błędy implementacyjne. W projekcie jest szczególnie użyteczny do pokazania, dlaczego warunek CFL jest ważny: po jego przekroczeniu rozwiązanie przestaje mieć sens fizyczny, pojawiają się oscylacje, a temperatura może przyjmować wartości nierealistyczne.

W zastosowaniach inżynierskich metoda jawna nadaje się do krótkich symulacji, małych siatek albo sytuacji, w których i tak potrzebujemy bardzo małego kroku czasu, aby uchwycić szybkie zjawiska. Nie jest najlepszym wyborem dla długiego chłodzenia pręta lub bardzo gęstej siatki, bo liczba kroków staje się zbyt duża.

### 8.2 Euler niejawny

Euler niejawny jest praktycznie ważny, ponieważ pozwala stabilnie liczyć procesy dyfuzji nawet dla dużych kroków czasowych. W problemach przewodzenia ciepła bywa to bardzo korzystne, gdy interesuje nas długoterminowy trend: na przykład czas dochodzenia układu do stanu bliskiego ustalonemu albo ogólny kierunek zanikania temperatury.

Cena za stabilność to tłumienie numeryczne. Profil temperatury może zostać wygładzony szybciej niż w rzeczywistym procesie. Dlatego Euler niejawny jest dobry, gdy ważniejsza jest odporność i stabilność obliczeń niż bardzo dokładny opis krótkich stanów przejściowych. W naszym przykładzie pręta stal–miedź metoda dobrze pokazuje, że duży krok czasu nie powoduje wybuchu rozwiązania, ale może zaniżyć maksymalną temperaturę i spłaszczyć profil.

### 8.3 Crank–Nicolson

Crank–Nicolson jest najbardziej wartościowy praktycznie jako kompromis między stabilnością, kosztem i dokładnością. Ma drugi rząd dokładności w czasie, a koszt jednego kroku jest podobny do Eulera niejawnego, ponieważ korzysta z tego samego typu faktoryzacji macierzy rzadkiej. Dzięki temu nadaje się do analizy przebiegu przejściowego, np. do badania, jak szybko ciepło przechodzi ze stali do miedzi i jak asymetria materiałowa wpływa na rozkład temperatury.

Trzeba jednak pamiętać, że Crank–Nicolson nie zawsze jest automatycznie „najbezpieczniejszy”. Przy nagłych skokach temperatury, niespójnych warunkach początkowo-brzegowych albo zbyt dużym kroku czasu może generować oscylacje. W praktyce oznacza to konieczność kontroli kroku $\Delta t$, wygładzenia danych początkowych albo wykonania kilku pierwszych kroków bardziej tłumiącą metodą, np. Eulerem niejawnym.

### 8.4 Podsumowanie wyboru metody

| Cel praktyczny | Najlepszy wybór | Uzasadnienie |
|---|---|---|
| Demonstracja stabilności i wpływu CFL | Euler jawny | najłatwiej pokazać, co dzieje się po złym dobraniu $\Delta t$ |
| Długie symulacje chłodzenia lub grzania | Euler niejawny | stabilny dla dużych kroków czasu |
| Dokładna analiza przejściowego przepływu ciepła | Crank–Nicolson | drugi rząd w czasie i umiarkowany koszt |
| Szybka walidacja kodu | Euler jawny + mały krok | prosty algorytm, łatwy do porównania z innymi metodami |
| Wyniki do interpretacji inżynierskiej | Crank–Nicolson lub Euler niejawny | zależnie od tego, czy ważniejsza jest dokładność profilu, czy odporność |

Dla modelu pręta stal–miedź najważniejsze znaczenie praktyczne ma możliwość przewidywania, gdzie powstają duże gradienty temperatury oraz jak różne materiały wpływają na transport ciepła. Stal, mając mniejszą przewodność cieplną, działa jak bariera spowalniająca przepływ energii, natomiast miedź szybciej rozprowadza i oddaje ciepło. Taki model można traktować jako uproszczony przykład analizy radiatorów, warstw izolacyjnych, elementów spawanych, przewodów cieplnych i konstrukcji złożonych z kilku materiałów.
